# Introduction

Eshel Dror (2026)

OFTeI (Orbits for the Extremely Impatient) is a partial linear marginalization scheme implemented in `orbitize!` for systems with relative astrometry data or absolute astrometry from Gaia Data Release 4. OFTeI works with the `MCMC` sampler to fit over fewer parameters while analytically calculating the rest of the parameters based on the data and conditional posterior probability. By fitting over fewer parameters, the posterior converges with much fewer samples. More information about OFTeI is in Blunt 2026b (in prep.)

In [ ]:
import os
import numpy as np
import multiprocessing as mp
import pandas as pd

import orbitize
from orbitize import read_input
from orbitize import basis
from orbitize import system
from orbitize import priors
from orbitize import marginalizer
from orbitize import sampler
from orbitize import plot
from orbitize import gaia

# OFTeI Relative Astrometry

OFTeI for systems with relative astrometry (Sep/PA and/or RA/Dec) is implemnted in `marginalizer.OFTeI_seppa`. The fitting must be done in the `basis.OFTeIBasis` basis, which (currently) only supports one companion with insignificant mass. Additionally, in this basis the period (`per1`) of the companion is fitted instead of the total mass (`mtot`). `marginalizer.OFTeI_seppa` is able to marginalize over the semimajor axis and position-angle of nodes. 

## Basic Usage

`orbitize!` will automatically set up the `OFTeI_seppa` marginalizer when initializing the `MCMC` sampler when the system is compatible. We will fit the same GJ504 system as the [MCMC tutorial](https://orbitize.readthedocs.io/en/latest/tutorials/MCMC_tutorial.html) as an example.

In [ ]:
filename = "{}/GJ504.csv".format(orbitize.DATADIR)

# system parameters
num_secondary_bodies = 1
total_mass = 1.75  # [Msol]
plx = 51.44  # [mas]
mass_err = 0.05  # [Msol]
plx_err = 0.12  # [mas]
tau_ref_epoch = 0

In [ ]:
# Read in data
data_table = read_input.read_file(filename)

In [ ]:
# Initialize System object which stores data & sets priors
mySystem = system.System(
    1,
    data_table,
    total_mass,
    plx,
    mass_err,
    plx_err,
    tau_ref_epoch=tau_ref_epoch,
    fitting_basis="OFTeIBasis",
)

In [ ]:
# Set priors
sma_prior_min = 0.1  # au
sma_prior_max = 500  # au
period_prior_min_yr = np.sqrt(sma_prior_min**3 / total_mass)  # [yr]
period_prior_max_yr = np.sqrt(sma_prior_max**3 / total_mass)  # [yr]
mySystem.sys_priors[mySystem.param_idx["sma1"]] = priors.LogUniformPrior(
    sma_prior_min, sma_prior_max
)
mySystem.sys_priors[mySystem.param_idx["per1"]] = priors.LogUniformPrior(
    period_prior_min_yr, period_prior_max_yr
)

In [ ]:
# MCMC parameters
num_temps = 5
num_walkers = 20
num_threads = mp.cpu_count()

In [ ]:
mySampler = sampler.MCMC(
    mySystem,
    num_temps=num_temps,
    num_walkers=num_walkers,
    num_threads=num_threads,
    marginalizers=None, # The marginalizer is automatically constructed when marginalizers=None (default)
)

In [ ]:
mySampler.marginalizers

In [ ]:
# Run a very short fit
total_orbits = num_walkers * 150
burn_steps = 50
thin = 5

mySampler.run_sampler(total_orbits, burn_steps=burn_steps, thin=thin)

In [ ]:
myResults = mySampler.results

In [ ]:
fig_seppa = plot.plot_corner(myResults)

In order to plot in the standard basis with total mass instead of period, set `use_standard_params=True`.

In [ ]:
fig_seppa_standard = plot.plot_corner(myResults, use_standard_params=True, range=[0.97]*8)

Compare the converge of the posterior to a fit without marginalization and with thrice the steps.

In [ ]:
myNoMargSampler = sampler.MCMC(
    mySystem,
    num_temps=num_temps,
    num_walkers=num_walkers,
    num_threads=num_threads,
    marginalizers=[], # No marginalization is done when marginalizers is an empty list
)

In [ ]:
myNoMargSampler.run_sampler(total_orbits*3, burn_steps=burn_steps*3, thin=thin)

In [ ]:
fig_no_marg = plot.plot_corner(myNoMargSampler.results, use_standard_params=True, range=[0.97]*8)

## Manual marginalizer setup

For more customization, the marginalizer can be set up automatically.

In [ ]:
myManSystem = system.System(
    1,
    data_table,
    total_mass,
    plx,
    mass_err,
    plx_err,
    tau_ref_epoch=tau_ref_epoch,
    fitting_basis="OFTeIBasis",
)

In [ ]:
# If your system had RA/Dec data, you would have to convert it to Sep/PA:
# myManSystem.convert_data_table_radec2seppa()
ind = data_table["quant_type"] == "seppa"
epochs = data_table["epoch"][ind]
sep = data_table["quant1"][ind]
pa = data_table["quant2"][ind]
sep_err = data_table["quant1_err"][ind]
pa_err = data_table["quant2_err"][ind]
# If the system had covariance, get it:
# covar = data_table["quant12_corr"][ind]
covar = np.zeros(len(epochs))

myMarg = marginalizer.OFTeI_seppa(
    epochs,
    sep,
    pa,
    sep_err,
    pa_err,
    covar,
    tau_ref_epoch,
)

In [ ]:
# Set priors
sma_prior_min = 0.1  # au
sma_prior_max = 500  # au
period_prior_min_yr = np.sqrt(sma_prior_min**3 / total_mass)  # [yr]
period_prior_max_yr = np.sqrt(sma_prior_max**3 / total_mass)  # [yr]
myManSystem.sys_priors[myManSystem.param_idx["sma1"]] = priors.LogUniformPrior(
    sma_prior_min, sma_prior_max
)
myManSystem.sys_priors[myManSystem.param_idx["per1"]] = priors.LogUniformPrior(
    period_prior_min_yr, period_prior_max_yr
)

In order to set a prior on system total mass, you must use a `ReweightPrior` to subtract the implied prior on `mtot` due to the `loguniform` priors on semimajor axis and period.

In [ ]:
implicit_mass_prior = priors.OFTeIMassPrior(
    sma_prior_min, sma_prior_max, period_prior_min_yr, period_prior_max_yr
)
mass_prior = priors.GaussianPrior(total_mass, mass_err)
reweight_prior = priors.OFTeIMassReweightPrior(implicit_mass_prior, mass_prior, mySystem.basis)

In [ ]:
myManSystem.reweight_priors.append(reweight_prior)

In [ ]:
myManSampler = sampler.MCMC(
    myManSystem,
    num_temps=num_temps,
    num_walkers=num_walkers,
    num_threads=num_threads,
    marginalizers=[myMarg],
)

In [ ]:
myManSampler.run_sampler(total_orbits, burn_steps=burn_steps, thin=thin)

In [ ]:
fig_manual = plot.plot_corner(myManSampler.results, use_standard_params=True)

## Analyzing OFTeI Posteriors

To perform other analysis on the posterior, you can convert it to standard basis. Make sure you perform this operation on a copy so you do not affect the `Results` object.

In [ ]:
post = myResults.post
param_arr = np.copy(post).T # Convert from orbits x params to params x orbits
standard_param_arr = myResults.basis.to_standard_basis(param_arr)
standard_post = standard_param_arr.T

In [ ]:
# Get mean total mass
np.mean(standard_post[:, myResults.standard_param_idx["mtot"]])

In [ ]:
# Get mean total mass through non-standard basis
sma = post[:, myResults.param_idx["sma1"]]
per = post[:, myResults.param_idx["per1"]]

mtot = (sma**3) / (per**2)

np.mean(mtot)

# OFTeI Gaia DR4

OFTeI for systems with absolute astrometry from Gaia Data Release 4 is implemnted in `marginalizer.OFTeI_DR4_gaia`. The fitting must be done in the `basis.OFTeIGaiaBasis` basis, which (currently) only supports one faint massive companion.
Additionally, in this basis:
1. The period (`per1`) of the companion is fitted instead of the primary mass (`m0`)
2. The mass fraction of the companion (`mfrac1`) is fitted instead of the companion  mass (`m1`) where the mass fraction is $\frac{M_0}{M_{tot}}=\frac{M_0}{M_0+M_1}$.
3. The parameters `u` and `v` are fitted instead of semimajor axis and position angle of nodes of the companion. $u=\pi a_p\cos{\Omega}$ and $u=\pi a_p\sin{\Omega}$ where $\pi$ is the parallax, $\Omega$ is the position-angle of nodes, and $a_p=\frac{M_0}{M_{tot}}a$ is the photometric semimajor axis with $a$ being the semimajor axis of the (faint) companion.

This allows `marginalizer.OFTeI_DR4_gaia` to marginalize over `u` & `v` (effectively semimajor axis and position angle of nodes), parallax (`plx`),  propermotion (`pm_ra` & `pm_dec`), and reference position offset (`alpha0`, `delta`).

## Marginalizer Construction

In [ ]:
# read solutions to pull pmra, pmdec, ra, dec that will be provided in the catalog.
solutions = pd.read_csv(os.path.join(orbitize.DATADIR, "injected_solutions.csv"))
target_file = "target_1.csv"
row = solutions[solutions["filename"] == target_file].iloc[0]
dr4_filepath = os.path.join(orbitize.DATADIR, target_file)
gaia_plx_guess = row["parallax_mas"]
gaia_plx_err = 0.5

# set up prior ranges, sampler params, ref epochs
m0_guess, m0_err = 1.0, 0.1
m1_lo, m1_hi = 1e-4, 0.1
sma_lo, sma_hi = 0.05, 50.0
ecc_lo, ecc_hi = 0.01, 0.99
GAIA_REF_MJD = 2457936.875-2400000.5
TAU_REF_EPOCH = 58849

dr4 = gaia.DR4LogProb(
    gaia_num=0,
    dr4_filepath=dr4_filepath,
    ref_epoch_mjd=GAIA_REF_MJD,
)

In [ ]:
myGaiaSystem = system.System(
    num_secondary_bodies=1, data_table=None,
    stellar_or_system_mass=m0_guess, plx=gaia_plx_guess,
    mass_err=m0_err, plx_err=gaia_plx_err,
    fit_secondary_mass=True, gaia=dr4, tau_ref_epoch=TAU_REF_EPOCH, fitting_basis="OFTeIGaiaBasis"
)

In [ ]:
#set priors in system
pi = myGaiaSystem.param_idx
myGaiaSystem.sys_priors[pi["ecc1"]] = priors.UniformPrior(ecc_lo, ecc_hi)
myGaiaSystem.sys_priors[pi["inc1"]] = priors.SinPrior()
myGaiaSystem.sys_priors[pi["aop1"]] = priors.UniformPrior(0, 2*np.pi)

sma_photo_hi = sma_hi * m1_hi / (m0_guess - 2 * m0_err)
u_hi = sma_photo_hi * (gaia_plx_guess + 2 * gaia_plx_err)
myGaiaSystem.sys_priors[pi["mfrac1"]] = priors.UniformPrior(m1_lo/(m0_guess + 2 * m0_err), m1_hi/(m0_guess - 2 * m0_err))
myGaiaSystem.sys_priors[pi["u"]] = priors.UniformPrior(-u_hi, u_hi)
myGaiaSystem.sys_priors[pi["v"]] = priors.UniformPrior(-u_hi, u_hi)
myGaiaSystem.sys_priors[pi["tau1"]] = priors.UniformPrior(0., 1.)
myGaiaSystem.sys_priors[pi["plx"]] = priors.GaussianPrior(gaia_plx_guess, gaia_plx_err, no_negatives=True)

period_prior_min_yr = np.sqrt(sma_lo**3 / m0_guess)
period_prior_max_yr = np.sqrt(sma_hi**3 / m0_guess)
myGaiaSystem.sys_priors[pi["per1"]] = priors.LogUniformPrior(period_prior_min_yr, period_prior_max_yr)

m0_prior = priors.GaussianPrior(m0_guess, m0_err, no_negatives=True)
pan_prior = priors.UniformPrior(0, 2*np.pi)
sma_prior = priors.LogUniformPrior(sma_lo, sma_hi)
myGaiaMarg = marginalizer.OFTeI_DR4_gaia(dr4, TAU_REF_EPOCH)

# implicit_mass_prior = priors.OFTeIMassPrior(sma_lo, sma_hi, period_prior_min_yr, period_prior_max_yr)
implicit_mass_prior = None # OFTeI for Gaia does not support a subtraction of an implicit mass prior yet, only addition of m0 prior
myGaiaSystem.reweight_priors.append(priors.OFTeIMassReweightPrior(implicit_mass_prior, m0_prior, myGaiaSystem.basis))
myGaiaSystem.reweight_priors.append(priors.OFTeIUVReweightPrior(sma_prior, pan_prior, myGaiaSystem.basis))

In [ ]:
n_walkers = 20
n_temps = 5
n_threads = 1 # Due to many matrix operations, OFTeI Gaia does not benefit from multiprocessing

In [ ]:
myGaiaSampler = sampler.MCMC(myGaiaSystem, num_temps=n_temps,
                    num_walkers=n_walkers, num_threads=n_threads, marginalizers=[myGaiaMarg])

In [ ]:
n_total_steps = n_walkers * 450
n_burn_steps = 200
thin = 5

myGaiaSampler.run_sampler(n_total_steps, burn_steps=n_burn_steps, thin=thin)

In [ ]:
myGaiaResults = myGaiaSampler.results

In [ ]:
# Plot corner in specified order
params = ["sma1", "ecc1", "inc1", "aop1", "pan1", "tau1", "plx", "pm_ra", "pm_dec", "alpha0", "delta0", "m1", "m0"]
fig_gaia = plot.plot_corner(myGaiaResults, use_standard_params=True, param_list=params, range=[0.97]*13)

## Reweight Priors and OFTeI posteriors

When analyzing posteriors, it is important to remember that, unless you set a reweight prior, the priors you set will be in the *marginaliation* basis. That is, if you do not set an `OFTeIUVReweightPrior`, the prior on semimajor axis and position-angle of nodes will be implied by your priors on parallax, u, and v (a uniform prior on those parameters will lead to $p(\pi,a,\Omega)\propto\pi a^2$). Setting a reweight prior subtracts this implied prior and adds your desired one.